# 09 — The Type System
**References:** Ramalho *Fluent Python* Ch. 8, 15 · PEP 484, 526, 544, 604, 612, 673 · mypy docs

## Narrative thread
```
Annotations -> TypeVar -> Generic -> Protocol -> Overload -> Literal -> TypedDict -> Runtime checks
```

## 1. Annotation basics and `typing` evolution

Python type hints are **annotations** — they have no runtime effect by default, but enable static analysis with `mypy`, `pyright`, and IDEs.

| Version | Change |
|---|---|
| 3.5 | `typing` module — `List`, `Dict`, `Optional`, `Union` |
| 3.9 | Built-in generics: `list[int]` instead of `List[int]` |
| 3.10 | `X \| Y` union syntax instead of `Union[X, Y]` |
| 3.10 | `match` statement (structural pattern matching) |
| 3.11 | `Self` type, `LiteralString`, `Never` |
| 3.12 | Type parameter syntax `type Alias = ...`, `class Foo[T]` |

In [ ]:
from __future__ import annotations  # postponed evaluation (PEP 563)

import sys
from typing import (
    TypeVar, Generic, Protocol, runtime_checkable,
    overload, Literal, TypedDict, get_type_hints,
    Callable, ParamSpec, Concatenate, TypeAlias,
    TYPE_CHECKING, Final, ClassVar, Annotated,
    NamedTuple, Never, Self, LiteralString,
)
from dataclasses import dataclass, field

# ── TypeVar and Generic ───────────────────────────────────────────────────
T = TypeVar('T')
S = TypeVar('S', bound='Comparable')

class Stack(Generic[T]):
    """Type-safe stack: Stack[int] can only hold ints."""
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        if not self._items:
            raise IndexError('pop from empty stack')
        return self._items.pop()

    def peek(self) -> T:
        return self._items[-1]

    def __len__(self) -> int:
        return len(self._items)

    def __repr__(self) -> str:
        return f'Stack({self._items})'

s: Stack[int] = Stack()
s.push(1); s.push(2); s.push(3)
print(f'Stack: {s}')
print(f'Pop:   {s.pop()}')
print(f'Peek:  {s.peek()}')

print()
# ── TypeVar with bound ───────────────────────────────────────────────────
def max_item(items: list[S]) -> S:
    """Return max of a list — only valid for Comparable types."""
    return max(items)

print(max_item([3, 1, 4, 1, 5, 9]))
print(max_item(['banana', 'apple', 'cherry']))

In [ ]:
# ── Protocol: structural subtyping ────────────────────────────────────────
@runtime_checkable
class Comparable(Protocol):
    def __lt__(self, other: Self) -> bool: ...
    def __le__(self, other: Self) -> bool: ...

@runtime_checkable
class Closeable(Protocol):
    def close(self) -> None: ...

class ResourcePool:
    def close(self) -> None:
        print('  Pool closed')

def cleanup(resource: Closeable) -> None:
    resource.close()

pool = ResourcePool()
print(f'isinstance(pool, Closeable): {isinstance(pool, Closeable)}')
cleanup(pool)  # no inheritance from Closeable — structural subtyping

print()
# ── @overload: multiple signatures ────────────────────────────────────────
@overload
def process(x: int) -> str: ...
@overload
def process(x: str) -> int: ...

def process(x: int | str) -> str | int:
    """Actual implementation — @overloads are type-check-only."""
    if isinstance(x, int):
        return str(x)
    return len(x)

r1: str = process(42)        # mypy knows this is str
r2: int = process('hello')   # mypy knows this is int
print(f'process(42):      {r1!r}  (type: {type(r1).__name__})')
print(f'process("hello"): {r2!r}  (type: {type(r2).__name__})')

print()
# ── Literal: restrict to specific values ────────────────────────────────
Mode = Literal['r', 'w', 'a', 'rb', 'wb']

def open_file(path: str, mode: Mode = 'r') -> None:
    print(f'open({path!r}, {mode!r})')

open_file('data.txt', 'r')
open_file('data.bin', 'rb')

In [ ]:
# ── TypedDict: typed dict with known keys ────────────────────────────────
class UserInfo(TypedDict):
    id:    int
    name:  str
    email: str

class UserInfoPartial(TypedDict, total=False):
    nickname: str
    age:      int

user: UserInfo = {'id': 1, 'name': 'Alice', 'email': 'alice@example.com'}
print(f'TypedDict: {user}')

print()
# ── NamedTuple: typed tuple ──────────────────────────────────────────────
class Point(NamedTuple):
    x: float
    y: float
    z: float = 0.0  # default value

    def distance_to(self, other: Point) -> float:
        return ((self.x-other.x)**2 + (self.y-other.y)**2 + (self.z-other.z)**2) ** 0.5

p1, p2 = Point(1.0, 2.0, 3.0), Point(4.0, 6.0, 3.0)
print(f'Point:    {p1}')
print(f'Distance: {p1.distance_to(p2):.3f}')
print(f'Unpack:   x={p1.x}, y={p1.y}, z={p1.z}')
x, y, z = p1  # tuples unpack

print()
# ── Annotated: attach metadata for validation/docs ───────────────────────
from typing import Annotated

Positive = Annotated[float, 'must be > 0']
Probability = Annotated[float, 'must be in [0, 1]']

def price_model(base: Positive, discount: Probability) -> Positive:
    return base * (1 - discount)

print(f'price_model(100, 0.2) = {price_model(100.0, 0.2)}')

# ParamSpec: preserve signature through decorator
P = ParamSpec('P')
R = TypeVar('R')

def log_call(fn: Callable[P, R]) -> Callable[P, R]:
    """ParamSpec preserves the decorated function's signature in type checkers."""
    import functools
    @functools.wraps(fn)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        print(f'  calling {fn.__name__}')
        return fn(*args, **kwargs)
    return wrapper

@log_call
def add(a: int, b: int) -> int:
    return a + b

result = add(3, 4)  # mypy knows result: int and args must be int
print(f'add(3, 4) = {result}')

## 2. `dataclasses` — the modern class factory

In [ ]:
from dataclasses import dataclass, field, fields, asdict, replace

@dataclass(frozen=True, order=True)   # immutable + comparison operators
class Money:
    amount:   float
    currency: str = 'USD'

    def __post_init__(self):
        if self.amount < 0:
            raise ValueError(f'amount must be >= 0, got {self.amount}')

    def convert(self, rate: float, currency: str) -> Money:
        return replace(self, amount=self.amount * rate, currency=currency)

m1 = Money(100.0)
m2 = Money(50.0, 'EUR')
print(f'm1: {m1}')
print(f'm2: {m2}')
print(f'm1 > m2 (same currency): {m1 > m2}')
print(f'm1 in EUR: {m1.convert(0.92, "EUR")}')

# frozen=True means hashable
print(f'hash(m1): {hash(m1)}')
prices = {m1: 'product A', m2: 'product B'}

@dataclass
class Graph:
    nodes:     list[str]        = field(default_factory=list)
    edges:     list[tuple]      = field(default_factory=list)
    directed:  bool             = False
    _cache:    dict             = field(default_factory=dict, repr=False, compare=False)

g = Graph(nodes=['A','B','C'], edges=[('A','B'),('B','C')])
print(f'Graph: {g}')
print(f'As dict: {asdict(g)}')

## 3. Key takeaways

| Concept | Rule |
|---|---|
| `TypeVar` | Generic type parameter; `bound=` restricts to subtypes |
| `Generic[T]` | Make a class generic — instantiate as `Stack[int]` |
| `Protocol` | Structural subtyping — `@runtime_checkable` enables `isinstance()` |
| `@overload` | Multiple signatures for type checkers only; one real implementation |
| `Literal` | Restrict to specific constant values |
| `TypedDict` | Typed dict — good for JSON payloads and kwargs |
| `ParamSpec` | Preserve function signature through decorators |
| `@dataclass(frozen=True)` | Immutable, hashable, comparable value object |

**Next:** notebook 10 — testing and design patterns.